# mROSE sequence generation demo

This notebook provides a GitHub-friendly walkthrough for checking the mROSE installation, verifying released checkpoints and running compact sequence-generation demos.

It is designed to run from a cloned copy of the repository. The generation cells are disabled by default so that the notebook can be opened safely on laptops or cloud notebooks without accidentally launching model inference.

<p align="center">
  <img src="../assets/mrose-icon.png" alt="mROSE icon" width="160">
</p>

<p align="center">
  <img src="../assets/mrose-figure1.png" alt="mROSE workflow" width="900">
</p>

## 1. Locate the repository

Run this notebook from `notebooks/` inside a local clone of `xiaoshaziydy/mrose`, or from the repository root. The following cell detects the project root and adds it to `sys.path`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "mrose").is_dir():
    PROJECT_ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "mrose").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 2. Installation notes

Clone with Git LFS enabled so the model checkpoints are downloaded as real `.pth` files rather than small pointer files.

```bash
git lfs install
git clone https://github.com/xiaoshaziydy/mrose.git
cd mrose
git lfs pull
conda env create -f environment.yml
conda activate mROSE
```

Alternatively, install the core pip dependencies:

```bash
pip install --extra-index-url https://download.pytorch.org/whl/cu118 -r requirements.txt
```

## 3. Dependency and checkpoint status

In [ ]:
import importlib.util

required_modules = ["numpy", "pandas", "torch", "Bio", "sklearn", "scipy", "tqdm"]
optional_modules = ["RNA", "ViennaRNA"]

def module_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

print("Dependency check")
for name in required_modules:
    print(f"  {name:10s}: {'OK' if module_available(name) else 'missing'}")
print("Optional RNA folding bindings")
for name in optional_modules:
    print(f"  {name:10s}: {'OK' if module_available(name) else 'missing'}")

In [ ]:
checkpoints = {
    "5utr": PROJECT_ROOT / "generation" / "5utr" / "Model.pth",
    "cds": PROJECT_ROOT / "generation" / "cds" / "Model.pth",
    "3utr": PROJECT_ROOT / "generation" / "3utr" / "Model.pth",
}

print("Checkpoint check")
for region, path in checkpoints.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  {region:4s}: {path.relative_to(PROJECT_ROOT)} ({size_mb:.1f} MB)")
    else:
        print(f"  {region:4s}: missing at {path.relative_to(PROJECT_ROOT)}")

## 4. Quick import check

In [ ]:
try:
    from mrose import (
        Model_5UTR,
        Model_CDS,
        Model_3UTR,
        GlobalmRNAMasterModel,
        compute_features_5utr,
        compute_features_3utr,
    )

    print("mROSE imports OK")
    print("5UTR feature length:", len(compute_features_5utr("ATGCGTACGTAG")))
    print("3UTR feature length:", len(compute_features_3utr("ATGCGTACGTAG")))
    print("Core classes:", Model_5UTR.__name__, Model_CDS.__name__, Model_3UTR.__name__, GlobalmRNAMasterModel.__name__)
except Exception as exc:
    print("Import check failed:", type(exc).__name__, exc)
    print("Install the environment first, then rerun this cell.")

## 5. Print ready-to-run generation commands

The repository includes a small launcher that prints commands for all supported generation demos and checks local dependencies/checkpoints.

In [ ]:
import subprocess

subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "demo_generate_sequences.py")],
    cwd=PROJECT_ROOT,
    check=False,
)

## 6. Run a compact generation demo

Set `RUN_DEMO = True` after dependencies and checkpoints are available. `cds` is the lightest default choice because the demo disables MFE scoring with `--mfe_weight 0`.

In [ ]:
RUN_DEMO = False
TASK = "cds"  # choose from: "5utr", "cds", "3utr", "all"

if RUN_DEMO:
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "demo_generate_sequences.py"), "--run", TASK],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print("Demo execution skipped. Set RUN_DEMO = True to run inference.")

## 7. Inspect generated outputs

After running a demo, generated files are written under `outputs/generation/`.

In [ ]:
output_root = PROJECT_ROOT / "outputs" / "generation"
if output_root.exists():
    for path in sorted(output_root.rglob("*")):
        if path.is_file():
            print(path.relative_to(PROJECT_ROOT))
else:
    print("No generation outputs found yet.")

## 8. Reproducibility checklist

- Use Git LFS before cloning or run `git lfs pull` after cloning.
- Verify released checkpoints with `shasum -a 256 -c MODEL_CHECKSUMS.sha256`.
- Keep full-scale datasets outside normal Git history.
- Use Zenodo or another archival repository for large public data releases.